# 세 가지 접근을 한 채점장에서 — 영상과 표회전형 역진자 swing-up 을 **세 가지 방법**으로 풀고, 같은 조건에서 나란히 돌린 영상과하나의 평가 표를 만든다.| | 접근 | 무엇으로 배우나 ||---|---|---|| 1 | **고전 제어** | 안 배운다 — 에너지 스윙업 + 고정 이득 || 2 | **모방학습** | 전문가 시연 (데이터 수를 바꿔가며) || 3 | **융합** | BC → DAgger → SAC fine-tune |## 채점장이 하나여야 하는 이유세 연구가 서로 다른 규약으로 만들어졌다. 그대로 두면 숫자를 나란히 쓸 수 없다.```고전 제어      rm.Loop 100 Hz, u [rad/s^2]모방학습       rm.Loop 100 Hz, 5프레임 스택 관측융합           자체 env 50 Hz, 5차원 관측, a in [-1,1]```그래서 **전부 실기의 루프로 데려온다** — `rotary_model.Loop`, 100 Hz,양자화 엔코더, 4 ms 지연, 140° 팔 정지. `arena.py` 가 그 채점장이고,`gallery.py` 가 거기서 영상과 표를 만든다.## 고전 제어에 대해 — PID는 혼자 스윙업을 못 한다매달린 막대에 `−Kx` 는 의미가 없다. 그래서 고정 이득 제어기는 실기와 **같은 에너지 스윙업상태기계**에 붙이고, **캐치 이득만** 다르게 한다.| | 이득 | 팔 되먹임 ||---|---|---|| `PID (rod only)` | `place2()` = `[k1, k2, 0, 0]` | **없음** || `4-state (expert)` | `place4()` | 있음 |차이는 **팔 항 하나**다. 그리고 그 결과가 표가 아니라 **영상에 보인다** —`place2` 는 팔을 되먹이지 않으니 `SwingUp` 의 수직 오프셋 추정기까지 죽고(그 이득이 `|K[2]|/…` 인데 0이다), 팔을 가운데로 당기는 것이 아무것도 없다.미리 돌려본 결과다.```PID (rod only)    t_up 2.43s  ...  팔 최대 141.0 deg   3판 전부 정지4-state (expert)  t_up 2.43s  ...  팔 최대  83.6 deg   정지 없음```**올리는 데까지는 똑같이 성공한다.** 그 뒤에 팔이 흘러가서 멈춘다.

## 1절 — 준비

In [ ]:
import os, sys, subprocess, pathlibREPO = pathlib.Path("/content/Pendulum_control")if not REPO.exists():    subprocess.run(["git", "clone", "--depth", "1",                    "https://github.com/jooyongsim/Pendulum_control.git", str(REPO)], check=True)else:    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)os.chdir(REPO)def find(rel):    hits = sorted(REPO.rglob(rel))    return hits[0].parent.relative_to(REPO).as_posix() if hits else NoneSIM = find("simulator/rotary_model.py")RLY = find("rl_yeen/rl_common.py")BCY = find("behavior_cloning_yeen/il_common.py")MINE = find("pipeline_yeen/gallery.py")PKGS = [p for p in (SIM, RLY, BCY, MINE) if p]for p in PKGS:    sys.path.insert(0, str(REPO / p))os.environ["PYTHONPATH"] = os.pathsep.join(str(REPO / p) for p in PKGS)os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"import numpy as np, pandas as pd, matplotlibmatplotlib.rcParams["figure.dpi"] = 110print("path ", PKGS)print("mine ", MINE)for name, p in (("simulator", SIM), ("behavior_cloning_yeen", BCY), ("pipeline_yeen", MINE)):    if p is None:        print(f"!! {name} 를 저장소에서 못 찾았다")import shutilprint("ffmpeg:", shutil.which("ffmpeg") or "없음 -> gif 로 저장된다")

In [ ]:
import arena, galleryprint(f"arena  100 Hz rm.Loop | 팔 정지 {arena.ARM_LIMIT_DEG:.0f} deg")for t, v in arena.TASKS.items():    print(f"  {t:<8s} {v['seconds']:.0f}초, 시작 {len(v['starts'])}개 -- {v['doc']}")print("suite robust:", [c["label"] for c in arena.SUITES["robust"]])

## 2절 — 데이터 수별 모방학습 모델 다시 만들기 (약 8분)데이터 수 비교 노트북은 학습한 체크포인트를 `/content/sweep` 에 뒀고 **저장소에 커밋하지않았다.** 그래서 영상에 넣으려면 다시 학습해야 한다.원래 설정 그대로다 — `SIZES = [40, 60, 80]`, 그리고 `bc_h5` 가 **206판 원본**이다.> ⚠️ 교사가 한 번도 올리지 못한 판은 가르칠 정답이 없어 버린다. 그래서 **넣은 값보다> 실제 채택 판 수가 적다** (40 → 31판 정도).> ⚠️ 학습 셀이 `behavior_cloning_yeen/checkpoints/bc_h5.npz` 를 **덮어쓴다.**> 그래서 원본을 먼저 빼둔다. (코랩의 복제본이라 저장소에는 영향이 없다.)

In [ ]:
import json, pathlib, numpy as npSWEEP = pathlib.Path("/content/sweep"); SWEEP.mkdir(exist_ok=True)CK = pathlib.Path(BCY) / "checkpoints"def rename(path, newname):    d = dict(np.load(path, allow_pickle=False))    meta = json.loads(str(d["meta_json"])); meta["name"] = newname    d["meta_json"] = json.dumps(meta)    np.savez_compressed(path, **d)def info(path):    r = json.loads(str(np.load(path, allow_pickle=False)["meta_json"]))["report"]    return r["episodes"], r["rmse"], r["rmse_swing"]shutil.copy(CK / "bc_h5.npz", SWEEP / "bc_206.npz")     # 206판 원본 보관rename(SWEEP / "bc_206.npz", "ep206")print("원본 보관 — %d판, RMSE %.3f, swing %.3f" % info(SWEEP / "bc_206.npz"))

In [ ]:
SIZES = [40, 60, 80]for n in SIZES:    print(f"\n{'='*62}\n  {n}판 수집 -> 학습\n{'='*62}")    data = f"{BCY}/data/expert_{n}.npz"    !PYTHONPATH={BCY} python {BCY}/collect_expert.py --episodes {n} --out {data}    !PYTHONPATH={BCY} python {BCY}/train_bc.py --data {data} --history 5    shutil.copy(CK / "bc_h5.npz", SWEEP / f"bc_{n}.npz")    kept = info(SWEEP / f"bc_{n}.npz")[0]    rename(SWEEP / f"bc_{n}.npz", f"ep{kept}")    print(f"  보관 -> bc_{n}.npz (실제 {kept}판)")print("\n보관된 모델")for p in sorted(SWEEP.glob("*.npz"), key=lambda q: info(q)[0]):    e, r, s = info(p)    print(f"  {p.name:<12} {e:>4}판   RMSE {r:.3f}   swing {s:.3f}")

## 3절 — 융합 체크포인트 확보`fusion.ipynb` 에서 만든 `.npz` 가 있어야 3번 접근이 영상에 들어간다. 세 가지 경우다.1. **저장소에 커밋했다** → 아래 셀이 바로 찾는다2. **그때 받은 `fusion_results.zip` 이 있다** → 왼쪽 📁 아이콘으로 업로드하고 아래 셀 실행3. **아직 안 돌렸다** → 그다음 셀에서 돌린다 (40~70분)없어도 **1·2번 접근 영상과 표는 그대로 만들어진다.**

In [ ]:
import pathlib, zipfile, globFUSE = pathlib.Path(MINE) / "checkpoints"; FUSE.mkdir(exist_ok=True)# fusion_results.zip 을 올려 두었으면 풀어서 넣는다for z in glob.glob("/content/fusion_results*.zip"):    with zipfile.ZipFile(z) as f:        for n in f.namelist():            if n.endswith(".npz"):                FUSE.joinpath(pathlib.Path(n).name).write_bytes(f.read(n))    print("풀었다:", z)found = sorted(p.name for p in FUSE.glob("*.npz"))print("융합 체크포인트:", found or "(없음 -- 다음 셀에서 만들거나, 건너뛰고 1·2번만)")

In [ ]:
# 없을 때만. 40~70분.if not list(FUSE.glob("sac_warm_*.npz")):    !python {MINE}/fusion.py --task swingup --steps 60000 --baseline --csv {MINE}/fusion_swingup.csvelse:    print("이미 있다 -- 건너뛴다")

## 4절 — 영상네 개를 만든다. 전부 **같은 시작, 같은 플랜트**다 — 패널 사이에 차이가 보이면 그건 오직 제어기 때문이다.| 파일 | 내용 ||---|---|| `classic_swingup.mp4` | PID (막대만) vs 4상태 전문가 || `imitation_swingup.mp4` | 데이터 수별 BC + DAgger || `fusion_swingup.mp4` | BC → DAgger → SAC warm / cold || `summary_swingup.mp4` | 접근별 대표 하나씩 |화면 구성 — **위** 진자 패널 (초록 부채꼴 = 직립 ±10°, 그 안에 들면 추가 초록으로),**가운데** `theta(t)`, **아래** `phi(t)` 와 빨간 **팔 한계선**. 팔 그래프가 핵심이다.

In [ ]:
from IPython.display import Video, displaymade = {}for g in ("classic", "imitation", "fusion", "summary"):    es = gallery.GROUPS[g](sweep_dir=str(SWEEP), ckpt_dir=str(FUSE))    if not es:        print(f"{g}: 대상 없음 -- 건너뜀"); continue    print(f"{g}: {[c.label for c in es]}")    p, sc = gallery.render(es, f"{MINE}/gallery/{g}_swingup.mp4",                           task="swingup", seconds=12.0, title=gallery.TITLES[g])    made[g] = p    print(f"  -> {p}  ({p.stat().st_size/1e6:.1f} MB)")

In [ ]:
for g, p in made.items():    print(g)    display(Video(str(p), embed=True, width=760))

## 5절 — 통합 평가 표모든 제어기를 **같은 보상(`full`)으로 채점**한다. 자기가 학습한 보상으로 잰 값은 측정이아니라 자기 평가다.직립 판정을 **±10°, ±5°, ±2°** 세 가지로 본다. 모방학습 모델이 **8.9° 틀어진 채 ±10°를100% 통과**한 적이 있어서, 한 기준만 보면 *버티는 정책* 과 *넘어지지만 않는 정책* 이 구분되지 않는다.| 지표 | 뜻 ||---|---|| `bias` | 마지막 5초 평균 각도 — 중심이 0에서 얼마나 틀어졌나 || `wobble` | 그 구간 표준편차 — 중심 주위로 얼마나 떠는가 || `t_up` | 처음 ±10°에 들어간 시각 || `arm` | 최대 \|φ\| (한계 140°) || `stop` | 호스트가 멈춘 판 수 |

In [ ]:
import pandas as pdrows = []for g in ("classic", "imitation", "fusion"):    es = gallery.GROUPS[g](sweep_dir=str(SWEEP), ckpt_dir=str(FUSE))    if es:        rows += gallery.rows_for(es, task="swingup", suite="clean", group=g)df = pd.DataFrame(rows).drop(columns=["cond"]).set_index(["group", "agent"])cols = ["ret", "hold10", "hold5", "hold2", "bias_deg", "wobble_deg", "t_up",        "arm_max_deg", "u_rms", "sat_pct", "stops"]df[cols].style.format("{:.1f}") \  .background_gradient(subset=["hold2"], cmap="RdYlGn", vmin=0, vmax=100) \  .background_gradient(subset=["arm_max_deg"], cmap="Oranges", vmin=40, vmax=140) \  .set_caption("100 Hz rm.Loop, 양자화 엔코더, 4 ms 지연 | 시작 3개 평균 | 보상 full 로 채점")

In [ ]:
# 모델 오차와 엔코더 영점까지 (b ±20%, 마찰 2배, 영점 2도)rows_r = []for g in ("classic", "imitation", "fusion"):    es = gallery.GROUPS[g](sweep_dir=str(SWEEP), ckpt_dir=str(FUSE))    if es:        rows_r += gallery.rows_for(es, task="swingup", suite="robust", group=g)tr = pd.DataFrame(rows_r)piv = tr.pivot_table(index=["group", "agent"], columns="cond", values="hold2", sort=False)piv = piv[[c["label"] for c in arena.SUITES["robust"]]]piv.style.format("{:.1f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=100) \   .set_caption("hold2 [%] — 마지막 5초를 ±2° 안에서 유지한 비율. "                "offset 2 = 엔코더 영점이 2° 틀린 조건")

In [ ]:
tr.to_csv(f"{MINE}/gallery/table_robust.csv", index=False)df.reset_index().to_csv(f"{MINE}/gallery/table_clean.csv", index=False)print("저장:", f"{MINE}/gallery/table_clean.csv", "/ table_robust.csv")

## 6절 — 내려받기

In [ ]:
import shutilshutil.make_archive("/content/gallery", "zip", f"{MINE}/gallery")print(*sorted(p.name for p in pathlib.Path(f"{MINE}/gallery").iterdir()), sep="\n")

In [ ]:
from google.colab import filesfiles.download("/content/gallery.zip")